In [18]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI()

In [19]:
from langchain_core.messages import HumanMessage

model.invoke(
    [
        HumanMessage(
            content="Hi! I'm Bob!",
        )
    ]
)

AIMessage(content='Hello Bob! How can I assist you today?', response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 13, 'total_tokens': 23}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run-7519f5d1-02ce-4f87-8fad-c31275c3ed51-0', usage_metadata={'input_tokens': 13, 'output_tokens': 10, 'total_tokens': 23})

In [20]:
model.invoke(
    [
        HumanMessage(
            content="What's my name?",
        )
    ]
)

AIMessage(content="I'm sorry, I do not know your name. Can you please tell me?", response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 12, 'total_tokens': 29}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run-efc960e4-2ab4-4f4f-97eb-f5317ed560b9-0', usage_metadata={'input_tokens': 12, 'output_tokens': 17, 'total_tokens': 29})

In [21]:
from langchain_core.messages import AIMessage

model.invoke(
    [
        HumanMessage(content="Hi! I'm Bob"),
        AIMessage(content="Hello Bob! How can I assist you today?"),
        HumanMessage(content="What's my name?"),
    ]
)

AIMessage(content="Your name is Bob. Is there anything else you'd like to know or talk about?", response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 35, 'total_tokens': 53}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run-dec20a24-ffe8-423f-94f3-87285331033e-0', usage_metadata={'input_tokens': 35, 'output_tokens': 18, 'total_tokens': 53})

In [31]:
from langchain_core.chat_history import (
    BaseChatMessageHistory,
    InMemoryChatMessageHistory,
)
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}


def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]


with_message_history = RunnableWithMessageHistory(
    model,
    get_session_history,
)

In [32]:
config = {
    "configurable": {
        "session_id": "abc2",
    }
}

In [33]:
response = with_message_history.invoke(
    [HumanMessage(content="Hi! I'm Bob")],
    config=config,
)

response.content

'Hi Bob! How can I assist you today?'

In [34]:
response = with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config,
)

response.content

'Your name is Bob.'

In [35]:
config = {"configurable": {"session_id": "abc3"}}

response = with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config,
)

response.content

"I'm sorry, I am an AI assistant and I do not have the ability to know your name."

In [36]:
config = {"configurable": {"session_id": "abc2"}}

response = with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config,
)

response.content

'Your name is Bob.'

In [37]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Answer all questions to the best of your ability.",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

chain = prompt | model

In [38]:
response = chain.invoke(
    {
        "messages": [
            HumanMessage(
                content="hi! I'm bob",
            )
        ]
    }
)

response.content

'Hello Bob! How can I assist you today?'

In [39]:
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
)

In [40]:
config = {
    "configurable": {
        "session_id": "abc5",
    }
}

In [41]:
response = with_message_history.invoke(
    [HumanMessage(content="Hi! I'm Jim")],
    config=config,
)

response.content

'Hello, Jim! How can I assist you today?'

In [42]:
response = with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config,
)

response.content

'Your name is Jim.'

In [43]:
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Answer all questions to the best of your ability in {language}.",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

chain = prompt | model

In [44]:
response = chain.invoke(
    {
        "messages": [
            HumanMessage(
                content="hi! I'm bob",
            )
        ],
        "language": "Spanish",
    }
)

response.content

'¡Hola, Bob! ¿En qué puedo ayudarte hoy?'

In [45]:
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages",
)

In [46]:
config = {
    "configurable": {
        "session_id": "abc11",
    }
}

In [47]:
response = with_message_history.invoke(
    {
        "messages": [
            HumanMessage(
                content="hi! I'm todd",
            )
        ],
        "language": "Spanish",
    },
    config=config,
)

response.content

'¡Hola Todd! ¿En qué puedo ayudarte hoy?'

In [48]:
response = with_message_history.invoke(
    {
        "messages": [
            HumanMessage(
                content="whats my name?",
            )
        ],
        "language": "Spanish",
    },
    config=config,
)

response.content

'Tu nombre es Todd. ¿Hay algo más en lo que pueda ayudarte?'

In [49]:
from langchain_core.messages import SystemMessage, trim_messages

trimmer = trim_messages(
    max_tokens=65,
    strategy="last",
    token_counter=model,
    include_system=True,
    allow_partial=False,
    start_on="human",
)

messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="hi! I'm bob"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
]

trimmer.invoke(messages)

[SystemMessage(content="you're a good assistant"),
 HumanMessage(content='whats 2 + 2'),
 AIMessage(content='4'),
 HumanMessage(content='thanks'),
 AIMessage(content='no problem!'),
 HumanMessage(content='having fun?'),
 AIMessage(content='yes!')]

In [50]:
from operator import itemgetter

from langchain_core.runnables import RunnablePassthrough

chain = (
    RunnablePassthrough.assign(messages=itemgetter("messages") | trimmer)
    | prompt
    | model
)

response = chain.invoke(
    {
        "messages": messages
        + [
            HumanMessage(
                content="what's my name?",
            )
        ],
        "language": "English",
    }
)
response.content

"I'm sorry, but I don't have access to personal information about users. How can I assist you today?"

In [51]:
response = chain.invoke(
    {
        "messages": messages
        + [
            HumanMessage(
                content="what math problem did i ask",
            )
        ],
        "language": "English",
    }
)
response.content

'You asked "what\'s 2 + 2?"'

In [52]:
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages",
)

config = {
    "configurable": {
        "session_id": "abc20",
    }
}

In [53]:
response = with_message_history.invoke(
    {
        "messages": messages
        + [
            HumanMessage(
                content="whats my name?",
            )
        ],
        "language": "English",
    },
    config=config,
)

response.content

"I'm sorry, I don't have access to your personal information."

In [54]:
response = with_message_history.invoke(
    {
        "messages": [
            HumanMessage(
                content="what math problem did i ask?",
            )
        ],
        "language": "English",
    },
    config=config,
)

response.content

"You haven't asked a specific math problem yet. Feel free to ask any math-related question, and I'll be happy to help!"

In [55]:
config = {"configurable": {"session_id": "abc15"}}
for r in with_message_history.stream(
    {
        "messages": [
            HumanMessage(
                content="hi! I'm todd. tell me a joke",
            )
        ],
        "language": "English",
    },
    config=config,
):
    print(
        r.content,
        end="|",
    )

|Hi| Todd|!| Sure|,| here|'s| a| joke| for| you|:

|Why| did| the| scare|crow| win| an| award|?
|Because| he| was| outstanding| in| his| field|!||